In [2]:
import polars as pl
import time

In [3]:
start_time = time.time()

lazy_df = pl.scan_csv('2019-Oct.csv')

head_df = lazy_df.head(5).collect()

end_time = time.time()

print(f"Time token to peak at 42 million rows: {end_time - start_time:.4f} seconds")
print('\nFirst 5 rows: ')
print(head_df)

Time token to peak at 42 million rows: 0.0065 seconds

First 5 rows: 
shape: (5, 9)
┌────────────┬────────────┬───────────┬───────────┬───┬──────────┬─────────┬───────────┬───────────┐
│ event_time ┆ event_type ┆ product_i ┆ category_ ┆ … ┆ brand    ┆ price   ┆ user_id   ┆ user_sess │
│ ---        ┆ ---        ┆ d         ┆ id        ┆   ┆ ---      ┆ ---     ┆ ---       ┆ ion       │
│ str        ┆ str        ┆ ---       ┆ ---       ┆   ┆ str      ┆ f64     ┆ i64       ┆ ---       │
│            ┆            ┆ i64       ┆ i64       ┆   ┆          ┆         ┆           ┆ str       │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪══════════╪═════════╪═══════════╪═══════════╡
│ 2019-10-01 ┆ view       ┆ 44600062  ┆ 210380745 ┆ … ┆ shiseido ┆ 35.79   ┆ 541312140 ┆ 72d76fde- │
│ 00:00:00   ┆            ┆           ┆ 959538772 ┆   ┆          ┆         ┆           ┆ 8bb3-4e00 │
│ UTC        ┆            ┆           ┆ 4         ┆   ┆          ┆         ┆           ┆ -8c23-a03 │
│      

# Converting CSV to Parquet

In [4]:
print("Converting 5.7GB CSV to Parquet... Grab a coffee, this will take a minuteor two!")
start_time = time.time()

pl.scan_csv('2019-Oct.csv').sink_parquet("2019-Oct.parquet")

end_time = time.time()

print(f"Conversion Complete! Time token: {end_time - start_time:.2f} seconds")

Converting 5.7GB CSV to Parquet... Grab a coffee, this will take a minuteor two!
Conversion Complete! Time token: 14.32 seconds


In [5]:
start_time = time.time()

head_df = pl.scan_parquet("2019-Oct.parquet").head(5).collect()

end_time = time.time()

print(f"Parquet scan time: {end_time - start_time:.4f} seconds")

Parquet scan time: 0.0078 seconds


# Data Exploration

We need to answer three massive questions before we can build an AI:

1. How many unique users are in this dataset?

2. How many shopping sessions occurred?

3. What is the breakdown of actions? (How many views vs. carts vs. purchases?)

In [6]:
start_time = time.time()

lf = pl.scan_parquet("2019-Oct.parquet")

metrics = lf.select([
  pl.len().alias('total rows'),
  pl.col('user_id').n_unique().alias('unique_users'),
  pl.col("user_session").n_unique().alias("unique_sessions"),
]).collect()

events = lf.group_by("event_type").len().sort("len", descending=True).collect()

end_time = time.time()

print(f"✅ Analysis complete in {end_time - start_time:.2f} seconds\n")
print("--- OVERALL METRICS ---")
print(metrics)
print("\n--- EVENT BREAKDOWN ---")
print(events)

✅ Analysis complete in 1.95 seconds

--- OVERALL METRICS ---
shape: (1, 3)
┌────────────┬──────────────┬─────────────────┐
│ total rows ┆ unique_users ┆ unique_sessions │
│ ---        ┆ ---          ┆ ---             │
│ u32        ┆ u32          ┆ u32             │
╞════════════╪══════════════╪═════════════════╡
│ 42448764   ┆ 3022290      ┆ 9244422         │
└────────────┴──────────────┴─────────────────┘

--- EVENT BREAKDOWN ---
shape: (3, 2)
┌────────────┬──────────┐
│ event_type ┆ len      │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ view       ┆ 40779399 │
│ cart       ┆ 926516   │
│ purchase   ┆ 742849   │
└────────────┴──────────┘


In [9]:
lf.head().collect()

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
str,str,i64,i64,str,str,f64,i64,str
"""2019-10-01 00:00:00 UTC""","""view""",44600062,2103807459595387724,null,"""shiseido""",35.79,541312140,"""72d76fde-8bb3-4e00-8c23-a032df…"
"""2019-10-01 00:00:00 UTC""","""view""",3900821,2053013552326770905,"""appliances.environment.water_h…","""aqua""",33.2,554748717,"""9333dfbd-b87a-4708-9857-633655…"
"""2019-10-01 00:00:01 UTC""","""view""",17200506,2053013559792632471,"""furniture.living_room.sofa""",null,543.1,519107250,"""566511c2-e2e3-422b-b695-cf8e6e…"
"""2019-10-01 00:00:01 UTC""","""view""",1307067,2053013558920217191,"""computers.notebook""","""lenovo""",251.74,550050854,"""7c90fc70-0e80-4590-96f3-13c02c…"
"""2019-10-01 00:00:04 UTC""","""view""",1004237,2053013555631882655,"""electronics.smartphone""","""apple""",1081.98,535871217,"""c6bd7419-2748-4c56-95b4-8cec9f…"


In [10]:
metrics.head()

total rows,unique_users,unique_sessions
u32,u32,u32
42448764,3022290,9244422


# Feature Engineering

For every single session, we want to calculate:

1. How many total actions did they take?

2. How many items did they view?

3. How many items did they put in their cart?

4. The Target Variable: Did they eventually make a purchase? (1 for yes, 0 for no).

In [11]:
start_time = time.time()

lf = pl.scan_parquet("2019-Oct.parquet")

sessions_lf = lf.group_by("user_session").agg([
  pl.col("user_id").first().alias("user_id"),
  pl.len().alias("total_actions"),
  
  (pl.col('event_type') == 'view').sum().alias("view_count"),
  (pl.col('event_type') == 'cart').sum().alias("cart_count"),
  (pl.col('event_type') == 'purchase').sum().alias("purchase_count"),
])

sessions_lf = sessions_lf.with_columns([
  (pl.col("purchase_count") > 0).cast(pl.Int8).alias("made_purchase")
])

sessions_df = sessions_lf.collect()

end_time = time.time()

print(f"✅ Feature Engineering complete in {end_time - start_time:.2f} seconds")
print(f"New Dataset Shape: {sessions_df.shape}")

print("\n--- Target Variable Distribution (Did they buy?) ---")
print(sessions_df["made_purchase"].value_counts())

✅ Feature Engineering complete in 2.87 seconds
New Dataset Shape: (9244422, 7)

--- Target Variable Distribution (Did they buy?) ---
shape: (2, 2)
┌───────────────┬─────────┐
│ made_purchase ┆ count   │
│ ---           ┆ ---     │
│ i8            ┆ u32     │
╞═══════════════╪═════════╡
│ 0             ┆ 8614862 │
│ 1             ┆ 629560  │
└───────────────┴─────────┘


In [12]:
sessions_df.head()

user_session,user_id,total_actions,view_count,cart_count,purchase_count,made_purchase
str,i64,u32,u32,u32,u32,i8
"""526c6832-718f-4ec9-bdb6-66d320…",526039147,6,6,0,0,0
"""8d917e0b-d7ff-4d89-9b90-805c2d…",559517500,7,7,0,0,0
"""cf5dcaae-141a-415c-9673-2e9ebc…",515607643,3,3,0,0,0
"""3bf0a38e-1cd0-4965-b95f-0936e6…",488355279,1,1,0,0,0
"""c330ed7b-9473-4551-99e8-25735c…",518507821,1,1,0,0,0


# Exploring Memory Usage

In [13]:
import sys

print('Current Memory usage:')
print(f"{sessions_df.to_pandas().memory_usage(deep=True).sum() / 1024**2:.2f} MB")


optimized_df = sessions_df.with_columns([
  pl.col("total_actions").cast(pl.Int32),
  pl.col("view_count").cast(pl.Int32),
  pl.col("cart_count").cast(pl.Int32),
  pl.col("purchase_count").cast(pl.Int32),
  pl.col("made_purchase").cast(pl.Int8),
])

print("\nOptimized Memory Usage:")
print(f"{optimized_df.to_pandas().memory_usage(deep=True).sum() / 1024**2:.2f} MB")

X = optimized_df.drop(['user_session', 'user_id', 'made_purchase', 'purchase_count', 'total_actions']).to_pandas()
y = optimized_df['made_purchase'].to_pandas()

print(f"Faetures shape: {X.shape}")

Current Memory usage:
609.42 MB

Optimized Memory Usage:
609.42 MB
Faetures shape: (9244422, 2)


In [14]:
X.head()

,view_count,cart_count
0,6,0
1,7,0
2,3,0
3,1,0
4,1,0


# Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

start_time = time.time()

model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

end_time = time.time()

print(f"Training complete in {end_time - start_time:.2f} seconds \n")

print("--- MODEL EVALUATION ---")
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))